1. 加载现有数据
2. 进行前置预筛选
3. 进行信息提取
4. 组织并输出

In [21]:
import tomllib
from pathlib import Path
from loguru import logger
CONFIG_PATH = "/Data_two/wyw/code/CETC_product/config.toml"

In [22]:
config = tomllib.load(Path(CONFIG_PATH).open("rb"))

In [23]:
def safe_get(d: dict, keys: list, default=None):
    """Safely get a nested value from a dictionary."""
    for key in keys:
        if isinstance(d, dict) and key in d:
            d = d[key]
        else:
            return default
    return d

In [48]:
ip = "/Data_two/wyw/code/CETC_product/data/prompts/NER_for_org.txt"
sorted(list(Path(ip).parent.glob(Path(ip).name)))

[PosixPath('/Data_two/wyw/code/CETC_product/data/prompts/NER_for_org.txt')]

In [42]:
from typing import Any, Generator, Tuple
import xopen
import orjson
import orjsonl
import yaml
def read_indata(
    inpath: Path, 
    encoding: str = "utf-8",
    skip_errors: bool = True
) -> Generator[Tuple[Path, Any], None, None]:
    """
    从文件中逐行读取 JSON 数据
    
    Args:
        inpath: 输入文件路径（支持通配符）
        encoding: 文件编码
        skip_errors: 是否跳过解析错误的行
    """
    # 获取匹配的文件列表
    inpaths = sorted(list(Path(inpath.parent).glob(inpath.name)))
    
    if not inpaths:
        logger.warning(f"未找到匹配的文件: {inpath}")
        return
    
    logger.info(f"找到 {len(inpaths)} 个文件")
    
    for file_idx, current_path in enumerate(inpaths):
        logger.info(f"[{file_idx + 1}/{len(inpaths)}] 处理文件: {current_path}")
        
        if not current_path.exists():
            logger.error(f"文件不存在: {current_path}")
            continue
            
        try:
            # 使用 with 确保文件正确关闭
            with xopen.xopen(current_path, "rt", encoding=encoding) as fin:
                for line_no, line in enumerate(fin, 1):
                    line = line.strip()
                    
                    # 跳过空行
                    if not line:
                        continue
                    
                    try:
                        data = orjson.loads(line)
                        yield current_path, data
                        
                    except orjson.JSONDecodeError as e:
                        if skip_errors:
                            logger.warning(
                                f"跳过无效 JSON (文件: {current_path.name}, "
                                f"行: {line_no}): {str(e)[:100]}"
                            )
                            continue
                        else:
                            raise ValueError(
                                f"JSON 解析失败 (文件: {current_path.name}, 行: {line_no}): {e}"
                            ) from e
                    
                    except Exception as e:
                        logger.error(f"处理行 {line_no} 时出错: {e}")
                        if not skip_errors:
                            raise
                            
        except Exception as e:
            logger.error(f"读取文件失败 {current_path}: {e}")
            if not skip_errors:
                raise

                
from cetc_product.data_model.entity import EntityType, DomainEnum, RegionEnum
target_domains = {d.value for d in DomainEnum.get_target_domains()}
target_regions = {r.value for r in RegionEnum.get_target_regions()}
def is_task1(record):
    entity_info = record.get("entity_classification", {}).get("result",{})
    if entity_info.get("type") != EntityType.ORGANIZATION.value:
        return False
    domains = safe_get(record, ["domain_and_region_classifier", "result", "domains"], None)
    if domains is None or (not(set(domains) & target_domains)):
        return False
    
    regions = safe_get(record, ["domain_and_region_classifier", "result", "regions"], None)
    if regions is None or (not(set(regions) & target_regions))  :
        return False
    return True

def get_input(data):
    infobox_info = safe_get(data, ["infoboxes_info"], None)
    if not infobox_info:
        infobox_info = ""
    else:
        infobox_info = yaml.safe_dump(infobox_info[0], allow_unicode=True, sort_keys=False)
    
    raw_text_path = Path(data.get("markdown_path",None))
    if raw_text_path.exists():
        fragment = raw_text_path.read_text(encoding="utf-8")[:1000]
    else:
        fragment = f"# {data['title']}\n\n{data['abstract']}"
    return {"wiki_text" : f"<infobox>{infobox_info}</infobox>\n\n{fragment}"}

In [5]:
# 准备提示词
from cetc_product.data_model.NER_for_org import OrganizationInfo
from cetc_product.tools.load_tool import load_prompt
prompt_path = "/Data_two/wyw/code/CETC_product/data/prompts/NER_for_org.txt"

prompt_temp = load_prompt(prompt_path, schema_define_cls=OrganizationInfo)

In [6]:
prompt_temp.pretty_print()

<system>
# 角色
专业的数据抽取专家，擅长从非结构化文本中精准地识别和提取特定信息，并将其结构化。

# 任务
是从一个机构的维基百科页面信息中，抽取出预先定义的实体信息。

# 核心规则 
1. 首要且最重要的任务是：首先判断输入的`title`是否指代一个具体、真实的组织机构，而非一个抽象概念、学科、学系或宽泛的类别。
    - 应抽取的目标：公司、政府部门、大学、研究所、军事单位、非政府组织等拥有实体存在和明确职能的单位
    - 正面示例：“中央情报局”、“清华大学”、“微软公司”
    - 应拒绝的目标：学科名称、专业领域、理论概念、职位类别等。
    - 负面示例：“信息工程学系”、“计算机科学”、“经济学”、“项目管理”
2. 按所给json schema输入标准json


# 思维引导:
- 如果 输入是具体机构：
  1.  尽力填充所有待抽取信息的字段。
- 如果 输入是抽象概念或学科（应被拒绝）：
  1.  “抽取信息结果”字段应该为“null”
2. 在“抽取理由字段”中简要说明原因，例如：“输入为具体机构，且... ”  或  “输入为学科名称，非具体机构...” 。

严格按照以下schema输出**合法json**，除此之外绝不得输出任何其他文字:
<SCHEMA>
{
  "$defs": {
    "BaseEntities": {
      "description": "定义所有需要抽取的实体类型的基础模型",
      "properties": {
        "alias": {
          "anyOf": [
            {
              "items": {
                "type": "string"
              },
              "type": "array"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "机构别名",
          "title": "Alias"

In [7]:
def extract_info(record, task):
    if task == "task2":
        raise NotImplementedError
    

In [47]:
Path("/mnt/1.json").suffix

'.json'

In [8]:
# 加载模型
from cetc_product.tools.llm_factory import llm_manager

In [9]:
from cetc_product.tools.json_parser import MyJSONParser


main_chain = prompt_temp | llm_manager["gpt_low"] | MyJSONParser(OrganizationInfo)

In [43]:
inpath = Path(config["DATA"]["IN"]["zhwiki_inpath"])
for p, data in read_indata(inpath):
    if is_task1(data):
        extract_result = main_chain.invoke(get_input(data))#extract_info(data, "task1")
        break
    
    # elif is_task2(data):
    #     extract_result = extract_info(data, "task2")
    
    ## OK 先对其进行提取
    ...

2025-11-15 22:37:59.700 | INFO     | __main__:read_indata:26 - 找到 5 个文件


2025-11-15 22:37:59.702 | INFO     | __main__:read_indata:29 - [1/5] 处理文件: /mnt/samba_shared/wyw/code/中电科_分类/data/domain_and_region/zhwiki/1/0.jsonl.zst


In [44]:
extract_result

ParseResult(parse=OrganizationInfo(consolidated_view=None, why='输入为对“資訊工程學系/CSIE”这一学科与学系类别的泛称描述，未指向某一所大学的具体院系组织，属于学科/类别而非具体机构，故不抽取。'), raw_text='{\n  "consolidated_view": null,\n  "why": "输入为对“資訊工程學系/CSIE”这一学科与学系类别的泛称描述，未指向某一所大学的具体院系组织，属于学科/类别而非具体机构，故不抽取。"\n}', error=None)

In [38]:
data.keys()

dict_keys(['infoboxes_info', 'categories_info', 'disambig_info', 'page_id', 'title', 'abstract', 'markdown_path', 'entity_classification', 'domain_and_region_classifier'])